In [1]:
# Import functions from helper python files.

import t5train_code, file_code, pairs_code, model_code
import importlib
from tqdm import tqdm
tqdm.pandas()

import pandas as pd

/home/user/Documents/ir/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-29 19:03:35.238086: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-29 19:03:35.273455: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-29 19:03:36.133828: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly diff

In [2]:
importlib.reload(file_code)
from file_code import *

In [3]:

json_train_labels_path = './data/task1_train_labels_2025.json'
json_test_labels_path = './data/task1_test_no_labels_2025.json'
train_files_path = './data/processed_train_langonly'
test_files_path = './data/processed_test_langonly'

In [9]:
import os, json

json_train_labels_path = './data/task1_train_labels_2025.json'
train_files_path = './data/processed_train_langonly'

# Load JSON
with open(json_train_labels_path, 'r') as f:
    train_labels = json.load(f)

# Build query and target lists
train_queries = [t.rstrip('.txt') for t in train_labels.keys()]
train_targets = []
for file in train_queries:
    train_targets.extend([t.rstrip('.txt') for t in train_labels[file + '.txt']])

# All referenced files (queries + targets)
referenced = set(train_queries + train_targets)

# All train files on disk
train_disk = set([f.rstrip('.txt') for f in os.listdir(train_files_path) if f.endswith('.txt')])

# Files that exist but are not labeled
unlabeled_train = sorted(list(train_disk - referenced))

print(f"Unlabeled train files: {len(unlabeled_train)}")
print(unlabeled_train[:20])  # show first 20


Unlabeled train files: 220
['000430', '001073', '001540', '001853', '002324', '002768', '003243', '003821', '004037', '004633', '005147', '005505', '005869', '007276', '007780', '008535', '008663', '009599', '009941', '010524']


In [10]:
import os, json

train_files_path = './data/processed_train_langonly'
test_files_path = './data/processed_test_langonly'
json_train_labels_path = './data/task1_train_labels_2025.json'
json_test_labels_path = './data/task1_test_no_labels_2025.json'

# Load
with open(json_train_labels_path, 'r') as f:
    train_labels = json.load(f)
with open(json_test_labels_path, 'r') as f:
    test_list = json.load(f)

train_queries = [t.rstrip('.txt') for t in train_labels.keys()]
train_targets = []
for file in train_queries:
    train_targets.extend([t.rstrip('.txt') for t in train_labels[file + '.txt']])

train_files = [f.rstrip('.txt') for f in os.listdir(train_files_path)]
test_files = [f.rstrip('.txt') for f in os.listdir(test_files_path)]

print("Train files on disk:", len(train_files))
print("Test files on disk:", len(test_files))
print("Train queries in JSON:", len(train_queries))
print("Train targets in JSON:", len(train_targets))
print("Test queries in JSON:", len(test_list))


Train files on disk: 7350
Test files on disk: 2159
Train queries in JSON: 1678
Train targets in JSON: 6881
Test queries in JSON: 400


In [3]:
files = get_files()

Reading raw data from text files. Generating files dataframe.


Read-in test files: 100%|██████████| 2159/2159 [00:01<00:00, 1492.21it/s]

Returning "files" df.


In [12]:
train_files = set([f.rstrip('.txt') for f in os.listdir(train_files_path)])
included = set(files[files['set']=='train']['filename'])
missing = train_files - included
print(len(missing), "missing train files")
list(missing)[:10]


220 missing train files


['036276',
 '016916',
 '015274',
 '024050',
 '086059',
 '020895',
 '047865',
 '048464',
 '095542',
 '030506']

In [4]:
files

,filename,set,query,cases,text
0,079311,train,False,[],"Strayer, J.\n: These are two applications unde..."
1,015880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S..."
2,042979,train,False,[],"Federal Court\nAngus Grant, for the applicant;..."
3,079847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...
4,076089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU..."
...,...,...,...,...,...
9284,087439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ..."
9285,056791,test,False,[],"[1]\n: When this matter first came before me, ..."
9286,063269,test,False,[],[1]\n: Jesse C. Stine (Mr. Stine) accompanied ...
9287,039286,test,False,[],<FRAGMENT_SUPPRESSED> CF 301)\nFederal Court\...


In [6]:
files_all = files

In [27]:
# row_true = files_all[files_all['query'] == True].sample(n=1)
# row_false = files_all[files_all['query'] == False].sample(n=1)

# # Combine them
# files = pd.concat([row_true, row_false])
# files

In [5]:
add_paragraphs(files)
files


Reading text from files. Extracting paragraphs based on regex pattern.


Extracting all paragraphs: 100%|██████████| 9289/9289 [00:05<00:00, 1742.97it/s]

Updated "files" df has with "paragraphs".


,filename,set,query,cases,text,paragraphs
0,079311,train,False,[],"Strayer, J.\n: These are two applications unde...","[[2] In April and May, 1993 the Department of ..."
1,015880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S...",[[1] : This is an application for a judicial r...
2,042979,train,False,[],"Federal Court\nAngus Grant, for the applicant;...",[[2] The guiding principle in respect of new e...
3,079847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...,[[1] : This is an application for judicial rev...
4,076089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU...",[[1] : This is an application for judicial rev...
...,...,...,...,...,...,...
9284,087439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ...","[[1] O'Reilly, J. : The Minister of National H..."
9285,056791,test,False,[],"[1]\n: When this matter first came before me, ...","[[1] : When this matter first came before me, ..."
9286,063269,test,False,[],[1]\n: Jesse C. Stine (Mr. Stine) accompanied ...,[[1] : Jesse C. Stine (Mr. Stine) accompanied ...
9287,039286,test,False,[],<FRAGMENT_SUPPRESSED> CF 301)\nFederal Court\...,[[1] : Ms. Bergeron asks the Court to set asid...


In [6]:
get_paragraphs_formatted(files)
files

Getting formatted paragraphs of length < 250 words.


Getting formatted paragraphs: 100%|██████████| 9289/9289 [00:06<00:00, 1369.90it/s]

Added formatted paragraphs to "files" df in "paragraphs_formatted".


,filename,set,query,cases,text,paragraphs,paragraphs_formatted
0,079311,train,False,[],"Strayer, J.\n: These are two applications unde...","[[2] In April and May, 1993 the Department of ...","[In April and May, 1993 the Department of the ..."
1,015880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S...",[[1] : This is an application for a judicial r...,[: This is an application for a judicial revie...
2,042979,train,False,[],"Federal Court\nAngus Grant, for the applicant;...",[[2] The guiding principle in respect of new e...,[The guiding principle in respect of new evide...
3,079847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...,[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...
4,076089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU...",[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...
...,...,...,...,...,...,...,...
9284,087439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ...","[[1] O'Reilly, J. : The Minister of National H...","[O'Reilly, J. : The Minister of National Healt..."
9285,056791,test,False,[],"[1]\n: When this matter first came before me, ...","[[1] : When this matter first came before me, ...","[: When this matter first came before me, I he..."
9286,063269,test,False,[],[1]\n: Jesse C. Stine (Mr. Stine) accompanied ...,[[1] : Jesse C. Stine (Mr. Stine) accompanied ...,[The Affidavit filed by Mr. G. Stine confirms ...
9287,039286,test,False,[],<FRAGMENT_SUPPRESSED> CF 301)\nFederal Court\...,[[1] : Ms. Bergeron asks the Court to set asid...,[: Ms. Bergeron asks the Court to set aside tw...


In [7]:
add_suppressed_sections(files)
files

Using regex and spacy (for long paragraphs) to extract and modify suppressed sections from paragraphs:


Extracting suppressed sections from paragraphs: 100%|██████████| 9289/9289 [08:34<00:00, 18.07it/s]  

Added suppressed sections to "suppressed_sections" field.


,filename,set,query,cases,text,paragraphs,paragraphs_formatted,suppressed_sections
0,079311,train,False,[],"Strayer, J.\n: These are two applications unde...","[[2] In April and May, 1993 the Department of ...","[In April and May, 1993 the Department of the ...",[]
1,015880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S...",[[1] : This is an application for a judicial r...,[: This is an application for a judicial revie...,[]
2,042979,train,False,[],"Federal Court\nAngus Grant, for the applicant;...",[[2] The guiding principle in respect of new e...,[The guiding principle in respect of new evide...,[]
3,079847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...,[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[]
4,076089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU...",[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[]
...,...,...,...,...,...,...,...,...
9284,087439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ...","[[1] O'Reilly, J. : The Minister of National H...","[O'Reilly, J. : The Minister of National Healt...",[[6] A person who wishes to have a drug patent...
9285,056791,test,False,[],"[1]\n: When this matter first came before me, ...","[[1] : When this matter first came before me, ...","[: When this matter first came before me, I he...",[]
9286,063269,test,False,[],[1]\n: Jesse C. Stine (Mr. Stine) accompanied ...,[[1] : Jesse C. Stine (Mr. Stine) accompanied ...,[The Affidavit filed by Mr. G. Stine confirms ...,[]
9287,039286,test,False,[],<FRAGMENT_SUPPRESSED> CF 301)\nFederal Court\...,[[1] : Ms. Bergeron asks the Court to set asid...,[: Ms. Bergeron asks the Court to set aside tw...,[]


In [8]:
# import re

# def clean_supressed_section_list(section_list):
#     """
#     Cleans each string in supressed_section list individually:
#     - Removes citation numbers like [1], [2], etc.
#     - Removes uppercase 'REFERENCE' and 'TARGETCASE' (with or without <>)
#     - Removes 'supra'
#     - Removes empty parentheses and extra punctuation
#     - Strips extra spaces
#     - Keeps only unique sentences per section
#     """
#     if not section_list:
#         return []
    
#     if not isinstance(section_list, list):
#         section_list = [section_list]
    
#     cleaned_list = []
#     for text in section_list:
#         # Remove citation numbers like [1], [23], etc.
#         text = re.sub(r"\[\d+\]", "", text)
#         # Remove uppercase REFERENCE/TARGETCASE (with or without < >)
#         text = re.sub(r"<*REFERENCE>*|<*TARGETCASE>*", "", text)
#         # Remove 'supra'
#         text = re.sub(r"\bsupra\b", "", text, flags=re.IGNORECASE)
#         # Remove empty parentheses
#         text = re.sub(r"\(\s*\)", "", text)
#         # Remove multiple punctuation marks and extra spaces
#         text = re.sub(r"[;,\s]{2,}", " ", text)
#         text = re.sub(r"\s+", " ", text)
#         # Strip leading/trailing spaces
#         text = text.strip()
#         if text:  # only keep non-empty strings
#             cleaned_list.append(text)
    
#     # Remove duplicates within this section while preserving order
#     seen = set()
#     unique_list = []
#     for item in cleaned_list:
#         if item not in seen:
#             seen.add(item)
#             unique_list.append(item)
    
#     return unique_list


# files["propositions"] = files["suppressed_sections"].apply(clean_supressed_section_list)
# files
add_propositions(files)

Using pre-trained t5 model to extract propositions from suppressed sections:


The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
Generating propositions from suppressed sections: 100%|██████████| 9289/9289 [1:29:30<00:00,  1.73it/s]  

Added propositions to "propositions" field.


In [9]:
files["propositions_en"] = files["propositions"]
files

,filename,set,query,cases,text,paragraphs,paragraphs_formatted,suppressed_sections,propositions,propositions_en
0,079311,train,False,[],"Strayer, J.\n: These are two applications unde...","[[2] In April and May, 1993 the Department of ...","[In April and May, 1993 the Department of the ...",[],[],[]
1,015880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S...",[[1] : This is an application for a judicial r...,[: This is an application for a judicial revie...,[],[],[]
2,042979,train,False,[],"Federal Court\nAngus Grant, for the applicant;...",[[2] The guiding principle in respect of new e...,[The guiding principle in respect of new evide...,[],[],[]
3,079847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...,[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[]
4,076089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU...",[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[]
...,...,...,...,...,...,...,...,...,...,...
9284,087439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ...","[[1] O'Reilly, J. : The Minister of National H...","[O'Reilly, J. : The Minister of National Healt...",[[6] A person who wishes to have a drug patent...,[A method of combining the active ingredients ...,[A method of combining the active ingredients ...
9285,056791,test,False,[],"[1]\n: When this matter first came before me, ...","[[1] : When this matter first came before me, ...","[: When this matter first came before me, I he...",[],[],[]
9286,063269,test,False,[],[1]\n: Jesse C. Stine (Mr. Stine) accompanied ...,[[1] : Jesse C. Stine (Mr. Stine) accompanied ...,[The Affidavit filed by Mr. G. Stine confirms ...,[],[],[]
9287,039286,test,False,[],<FRAGMENT_SUPPRESSED> CF 301)\nFederal Court\...,[[1] : Ms. Bergeron asks the Court to set asid...,[: Ms. Bergeron asks the Court to set aside tw...,[],[],[]


In [10]:
files.to_csv('embeddings_new_2_with_prop_afterprop.csv', index=False) 

In [ ]:
import pandas as pd
import ast

def revert_dataframe_columns(df):
    for col in df.columns:
        def try_eval(x):
            if isinstance(x, str):
                try:
                    return ast.literal_eval(x)
                except (ValueError, SyntaxError):
                    return x
            return x
        df[col] = df[col].apply(try_eval)
    return df

files = pd.read_csv('embeddings_new_2_with_prop_afterprop.csv')
files = revert_dataframe_columns(files)


In [6]:
files

,filename,set,query,cases,text,paragraphs,paragraphs_formatted,suppressed_sections,propositions,propositions_en
0,79311,train,False,[],"Strayer, J.\n: These are two applications unde...","[[2] In April and May, 1993 the Department of ...","[In April and May, 1993 the Department of the ...",[],[],[]
1,15880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S...",[[1] : This is an application for a judicial r...,[: This is an application for a judicial revie...,[],[],[]
2,42979,train,False,[],"Federal Court\nAngus Grant, for the applicant;...",[[2] The guiding principle in respect of new e...,[The guiding principle in respect of new evide...,[],[],[]
3,79847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...,[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[]
4,76089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU...",[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[]
...,...,...,...,...,...,...,...,...,...,...
9284,87439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ...","[[1] O'Reilly, J. : The Minister of National H...","[O'Reilly, J. : The Minister of National Healt...",[[6] A person who wishes to have a drug patent...,[A person who wishes to have a drug patent pro...,[A person who wishes to have a drug patent pro...
9285,56791,test,False,[],"[1]\n: When this matter first came before me, ...","[[1] : When this matter first came before me, ...","[: When this matter first came before me, I he...",[],[],[]
9286,63269,test,False,[],[1]\n: Jesse C. Stine (Mr. Stine) accompanied ...,[[1] : Jesse C. Stine (Mr. Stine) accompanied ...,[The Affidavit filed by Mr. G. Stine confirms ...,[],[],[]
9287,39286,test,False,[],<FRAGMENT_SUPPRESSED> CF 301)\nFederal Court\...,[[1] : Ms. Bergeron asks the Court to set asid...,[: Ms. Bergeron asks the Court to set aside tw...,[],[],[]


In [11]:
add_sentences(files)

Using spacy to extract sentences of char length > 25 from paragraphs (parallelized)


100%|██████████| 9289/9289 [14:50<00:00, 10.43it/s] 

Added lists of sentences to "sentences".


,filename,set,query,cases,text,paragraphs,paragraphs_formatted,suppressed_sections,propositions,propositions_en,sentences
0,079311,train,False,[],"Strayer, J.\n: These are two applications unde...","[[2] In April and May, 1993 the Department of ...","[In April and May, 1993 the Department of the ...",[],[],[],"[In April and May, 1993 the Department of the ..."
1,015880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S...",[[1] : This is an application for a judicial r...,[: This is an application for a judicial revie...,[],[],[],[: This is an application for a judicial revie...
2,042979,train,False,[],"Federal Court\nAngus Grant, for the applicant;...",[[2] The guiding principle in respect of new e...,[The guiding principle in respect of new evide...,[],[],[],[The guiding principle in respect of new evide...
3,079847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...,[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[],[: This is an application for judicial review ...
4,076089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU...",[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[],[: This is an application for judicial review ...
...,...,...,...,...,...,...,...,...,...,...,...
9284,087439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ...","[[1] O'Reilly, J. : The Minister of National H...","[O'Reilly, J. : The Minister of National Healt...",[[6] A person who wishes to have a drug patent...,[A method of combining the active ingredients ...,[A method of combining the active ingredients ...,"[O'Reilly, J. : The Minister of National Healt..."
9285,056791,test,False,[],"[1]\n: When this matter first came before me, ...","[[1] : When this matter first came before me, ...","[: When this matter first came before me, I he...",[],[],[],"[: When this matter first came before me, I he..."
9286,063269,test,False,[],[1]\n: Jesse C. Stine (Mr. Stine) accompanied ...,[[1] : Jesse C. Stine (Mr. Stine) accompanied ...,[The Affidavit filed by Mr. G. Stine confirms ...,[],[],[],[: Jesse C. Stine (Mr. Stine) accompanied his ...
9287,039286,test,False,[],<FRAGMENT_SUPPRESSED> CF 301)\nFederal Court\...,[[1] : Ms. Bergeron asks the Court to set asid...,[: Ms. Bergeron asks the Court to set aside tw...,[],[],[],[: Ms. Bergeron asks the Court to set aside tw...


In [12]:
files.to_csv('embeddings_new_2_with_prop_aftersent.csv', index=False)

In [13]:
files["sentences_en"] = files["sentences"]
files

,filename,set,query,cases,text,paragraphs,paragraphs_formatted,suppressed_sections,propositions,propositions_en,sentences,sentences_en
0,079311,train,False,[],"Strayer, J.\n: These are two applications unde...","[[2] In April and May, 1993 the Department of ...","[In April and May, 1993 the Department of the ...",[],[],[],"[In April and May, 1993 the Department of the ...","[In April and May, 1993 the Department of the ..."
1,015880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S...",[[1] : This is an application for a judicial r...,[: This is an application for a judicial revie...,[],[],[],[: This is an application for a judicial revie...,[: This is an application for a judicial revie...
2,042979,train,False,[],"Federal Court\nAngus Grant, for the applicant;...",[[2] The guiding principle in respect of new e...,[The guiding principle in respect of new evide...,[],[],[],[The guiding principle in respect of new evide...,[The guiding principle in respect of new evide...
3,079847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...,[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[],[: This is an application for judicial review ...,[: This is an application for judicial review ...
4,076089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU...",[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[],[: This is an application for judicial review ...,[: This is an application for judicial review ...
...,...,...,...,...,...,...,...,...,...,...,...,...
9284,087439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ...","[[1] O'Reilly, J. : The Minister of National H...","[O'Reilly, J. : The Minister of National Healt...",[[6] A person who wishes to have a drug patent...,[A method of combining the active ingredients ...,[A method of combining the active ingredients ...,"[O'Reilly, J. : The Minister of National Healt...","[O'Reilly, J. : The Minister of National Healt..."
9285,056791,test,False,[],"[1]\n: When this matter first came before me, ...","[[1] : When this matter first came before me, ...","[: When this matter first came before me, I he...",[],[],[],"[: When this matter first came before me, I he...","[: When this matter first came before me, I he..."
9286,063269,test,False,[],[1]\n: Jesse C. Stine (Mr. Stine) accompanied ...,[[1] : Jesse C. Stine (Mr. Stine) accompanied ...,[The Affidavit filed by Mr. G. Stine confirms ...,[],[],[],[: Jesse C. Stine (Mr. Stine) accompanied his ...,[: Jesse C. Stine (Mr. Stine) accompanied his ...
9287,039286,test,False,[],<FRAGMENT_SUPPRESSED> CF 301)\nFederal Court\...,[[1] : Ms. Bergeron asks the Court to set asid...,[: Ms. Bergeron asks the Court to set aside tw...,[],[],[],[: Ms. Bergeron asks the Court to set aside tw...,[: Ms. Bergeron asks the Court to set aside tw...


In [14]:
add_quotes(files)
add_entities(files)
add_strings_sets(files)
files.to_csv('embeddings_new_2_with_prop_afterstring.csv', index=False)
files

Using regex to extract quotations from suppressed sections:


Extracting quotes from suppressed sections:: 100%|██████████| 9289/9289 [00:00<00:00, 12697.13it/s]

Added quotes to the "quotes" field for query cases.
Using spacy to extract noun entities from English sentences (parallelized).



100%|██████████| 9289/9289 [25:31<00:00,  6.07it/s]  

Added entity strings to "entity_string" and entities as sets to "entity_set".
Extracting case word strings (for tfidf) and sets (for case jaccard):



Extracting case string and sets from sentences_en:: 100%|██████████| 9289/9289 [01:54<00:00, 81.26it/s] 


Added case strings and sets.


,filename,set,query,cases,text,paragraphs,paragraphs_formatted,suppressed_sections,propositions,propositions_en,sentences,sentences_en,quotes,entity_string,entity_set,sentences_en_string,sentences_en_set
0,079311,train,False,[],"Strayer, J.\n: These are two applications unde...","[[2] In April and May, 1993 the Department of ...","[In April and May, 1993 the Department of the ...",[],[],[],"[In April and May, 1993 the Department of the ...","[In April and May, 1993 the Department of the ...",[],departmentsecretary state accessinformationact...,"{, departmentsecretary, sociétégamma, governme...",april depart secretari state receiv request in...,"{mere, exempt, express, futur, manag, head, ap..."
1,015880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S...",[[1] : This is an application for a judicial r...,[: This is an application for a judicial revie...,[],[],[],[: This is an application for a judicial revie...,[: This is an application for a judicial revie...,[],visa applicant applicant applicant dubai perso...,"{personalbackground, adaptability, court, immi...",applic judici review decis visa offic date oct...,"{show, assess, passag, unit, unreason, date, p..."
2,042979,train,False,[],"Federal Court\nAngus Grant, for the applicant;...",[[2] The guiding principle in respect of new e...,[The guiding principle in respect of new evide...,[],[],[],[The guiding principle in respect of new evide...,[The guiding principle in respect of new evide...,[],riskassessment prra prra immigrationrefugeepr...,"{, personalinformationform, imm, applicant, ku...",guid principl respect new evid submit preremov...,"{show, tortur, ms, express, villag, divis, unc..."
3,079847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...,[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[],[: This is an application for judicial review ...,[: This is an application for judicial review ...,[],conventionrefugeedeterminationdivisionimmigrat...,"{, immigrationact, division, conventionrefugee...",applic judici review decis convent refuge dete...,"{prob, includ, juli, mere, assess, unit, objec..."
4,076089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU...",[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[],[: This is an application for judicial review ...,[: This is an application for judicial review ...,[],immigrationrefugeeprotectionact punjab pakist...,"{, pakistanmuslimleague, pakistanmuslimleagueg...",applic judici review pursuant subsect immigr r...,"{interpret, mere, assess, death, tortur, quash..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9284,087439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ...","[[1] O'Reilly, J. : The Minister of National H...","[O'Reilly, J. : The Minister of National Healt...",[[6] A person who wishes to have a drug patent...,[A method of combining the active ingredients ...,[A method of combining the active ingredients ...,"[O'Reilly, J. : The Minister of National Healt...","[O'Reilly, J. : The Minister of National Healt...",[the medicine itself or a claim for the use of...,nationalhealthwelfare biovail patentedmedicin...,"{, decisiontherapeuticproductsdirectorate, wel...",oreilli j minist nation health welfar deni bio...,"{cattl, deliveri, mere, substanc, administ, wi..."
9285,056791,test,False,[],"[1]\n: When this matter first came before me, ...","[[1] : When this matter first came before me, ...","[: When this matter first came before me, I he...",[],[],[],"[: When this matter first came before me, I he...","[: When this matter first came before me, I he...",[],toblerone kraft toblerone canada kraft canadia...,"{canadian, , application, copyrightact, tobler...",matter came held sell deal certain line tobler...,"{deliveri, show, straightforward, chosen, clem..."
9286,063269,test,False,[],[1]\n: Jesse C. Stine (Mr. Stine) accompanied .

In [15]:
add_set_lists(files)
add_judge_name(files)
add_year(files)
files.to_csv('embeddings_new_2_with_prop_addyear.csv', index=False)
files

Extracting set lists:


Extracting set list from propositions_en:: 100%|██████████| 9289/9289 [00:02<00:00, 3821.04it/s]

Added set lists.
Using regex to extract judge surname from first paragraphs:



Extracting judge surname from paragraphs: 100%|██████████| 9289/9289 [00:00<00:00, 630649.41it/s]

Added judge name to "judge" field.
Using string search to find year:



Extracting most recent year from file text: 100%|██████████| 9289/9289 [00:02<00:00, 3609.79it/s]

Added year to files.


,filename,set,query,cases,text,paragraphs,paragraphs_formatted,suppressed_sections,propositions,propositions_en,...,quotes,entity_string,entity_set,sentences_en_string,sentences_en_set,sentences_en_set_list,paragraphs_formatted_en_set_list,propositions_en_set_list,judge,year
0,079311,train,False,[],"Strayer, J.\n: These are two applications unde...","[[2] In April and May, 1993 the Department of ...","[In April and May, 1993 the Department of the ...",[],[],[],...,[],departmentsecretary state accessinformationact...,"{, departmentsecretary, sociétégamma, governme...",april depart secretari state receiv request in...,"{mere, exempt, express, futur, manag, head, ap...","[{contract, translat, inform, receiv, individu...","[{translat, gave, provid, understand, question...",[],None,1994
1,015880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S...",[[1] : This is an application for a judicial r...,[: This is an application for a judicial revie...,[],[],[],...,[],visa applicant applicant applicant dubai perso...,"{personalbackground, adaptability, court, immi...",applic judici review decis visa offic date oct...,"{show, assess, passag, unit, unreason, date, p...","[{octob, judici, review, offic, decis, applic,...","[{octob, judici, fact, review, offic, decis, a...",[],None,2008
2,042979,train,False,[],"Federal Court\nAngus Grant, for the applicant;...",[[2] The guiding principle in respect of new e...,[The guiding principle in respect of new evide...,[],[],[],...,[],riskassessment prra prra immigrationrefugeepr...,"{, personalinformationform, imm, applicant, ku...",guid principl respect new evid submit preremov...,"{show, tortur, ms, express, villag, divis, unc...","[{justic, evid, judith, snider, assess, princi...","[{justic, evid, judith, snider, assess, princi...",[],None,2009
3,079847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...,[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[],...,[],conventionrefugeedeterminationdivisionimmigrat...,"{, immigrationact, division, conventionrefugee...",applic judici review decis convent refuge dete...,"{prob, includ, juli, mere, assess, unit, objec...","[{board, juli, immigr, review, base, c, decis,...","[{board, juli, immigr, review, unit, truck, ba...",[],None,2002
4,076089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU...",[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[],...,[],immigrationrefugeeprotectionact punjab pakist...,"{, pakistanmuslimleague, pakistanmuslimleagueg...",applic judici review pursuant subsect immigr r...,"{interpret, mere, assess, death, tortur, quash...","[{immigr, review, alexand, c, decis, render, g...","[{immigr, review, alexand, c, decis, render, g...",[],None,2010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9284,087439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ...","[[1] O'Reilly, J. : The Minister of National H...","[O'Reilly, J. : The Minister of National Healt...",[[6] A person who wishes to have a drug patent...,[A method of combining the active ingredients ...,[A method of combining the active ingredients ...,...,[the medicine itself or a claim for the use of...,nationalhealthwelfare biovail patentedmedicin...,"{, decisiontherapeuticproductsdirectorate, wel...",oreilli j minist nation health welfar deni bio...,"{cattl, deliveri, mere, substanc, administ, wi...","[{nation, medicin, afford, complianc, patent, ...","[{complianc, medicin, patent, welfar, minist, ...","[{activ, effect, patent, substanc, similar, dr...",O'Reilly,2005
9285,056791,test,False,[],"[1]\n: When this matter first came before me, ...","[[1] : When this matter first came before me, ...","[: When this matter first came before me, I he...",[],[],[],...,[],toblerone kraft toblerone canada kraft canadia...,"{canadian, , application, copyrightact, tobler...",matter came held

In [16]:
files

,filename,set,query,cases,text,paragraphs,paragraphs_formatted,suppressed_sections,propositions,propositions_en,...,quotes,entity_string,entity_set,sentences_en_string,sentences_en_set,sentences_en_set_list,paragraphs_formatted_en_set_list,propositions_en_set_list,judge,year
0,079311,train,False,[],"Strayer, J.\n: These are two applications unde...","[[2] In April and May, 1993 the Department of ...","[In April and May, 1993 the Department of the ...",[],[],[],...,[],departmentsecretary state accessinformationact...,"{, departmentsecretary, sociétégamma, governme...",april depart secretari state receiv request in...,"{mere, exempt, express, futur, manag, head, ap...","[{contract, translat, inform, receiv, individu...","[{translat, gave, provid, understand, question...",[],None,1994
1,015880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S...",[[1] : This is an application for a judicial r...,[: This is an application for a judicial revie...,[],[],[],...,[],visa applicant applicant applicant dubai perso...,"{personalbackground, adaptability, court, immi...",applic judici review decis visa offic date oct...,"{show, assess, passag, unit, unreason, date, p...","[{octob, judici, review, offic, decis, applic,...","[{octob, judici, fact, review, offic, decis, a...",[],None,2008
2,042979,train,False,[],"Federal Court\nAngus Grant, for the applicant;...",[[2] The guiding principle in respect of new e...,[The guiding principle in respect of new evide...,[],[],[],...,[],riskassessment prra prra immigrationrefugeepr...,"{, personalinformationform, imm, applicant, ku...",guid principl respect new evid submit preremov...,"{show, tortur, ms, express, villag, divis, unc...","[{justic, evid, judith, snider, assess, princi...","[{justic, evid, judith, snider, assess, princi...",[],None,2009
3,079847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...,[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[],...,[],conventionrefugeedeterminationdivisionimmigrat...,"{, immigrationact, division, conventionrefugee...",applic judici review decis convent refuge dete...,"{prob, includ, juli, mere, assess, unit, objec...","[{board, juli, immigr, review, base, c, decis,...","[{board, juli, immigr, review, unit, truck, ba...",[],None,2002
4,076089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU...",[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[],...,[],immigrationrefugeeprotectionact punjab pakist...,"{, pakistanmuslimleague, pakistanmuslimleagueg...",applic judici review pursuant subsect immigr r...,"{interpret, mere, assess, death, tortur, quash...","[{immigr, review, alexand, c, decis, render, g...","[{immigr, review, alexand, c, decis, render, g...",[],None,2010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9284,087439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ...","[[1] O'Reilly, J. : The Minister of National H...","[O'Reilly, J. : The Minister of National Healt...",[[6] A person who wishes to have a drug patent...,[A method of combining the active ingredients ...,[A method of combining the active ingredients ...,...,[the medicine itself or a claim for the use of...,nationalhealthwelfare biovail patentedmedicin...,"{, decisiontherapeuticproductsdirectorate, wel...",oreilli j minist nation health welfar deni bio...,"{cattl, deliveri, mere, substanc, administ, wi...","[{nation, medicin, afford, complianc, patent, ...","[{complianc, medicin, patent, welfar, minist, ...","[{activ, effect, patent, substanc, similar, dr...",O'Reilly,2005
9285,056791,test,False,[],"[1]\n: When this matter first came before me, ...","[[1] : When this matter first came before me, ...","[: When this matter first came before me, I he...",[],[],[],...,[],toblerone kraft toblerone canada kraft canadia...,"{canadian, , application, copyrightact, tobler...",matter came held

In [17]:
import os
import torch
import logging
import numpy as np
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

# ---------------- Helper Classes and Functions ---------------- #

class PropositionDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]


def get_embedding_list(dataset, tokenizer, model, device, batch_size=32):
    dataloader = DataLoader(dataset, batch_size=batch_size)
    embeddings = torch.Tensor().to(device)

    for batch in dataloader:
        inputs = tokenizer(
            batch, 
            return_tensors='pt', 
            padding=True, 
            truncation=True, 
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        embeddings = torch.cat((embeddings, outputs.last_hidden_state[:, 0, :]), dim=0)

    embeddings_np = embeddings.cpu().numpy()
    return [emb for emb in embeddings_np]


def init_model():
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    return tokenizer, model, device


# ---------------- Embedding Generation Functions ---------------- #

def generate_sentence_embeddings(files, save_path):
    logging.info("Generating sentence embeddings...")
    tokenizer, model, device = init_model()
    tqdm.pandas(desc="Getting embeddings for sentences_en")

    files['embeddings_sentences_en'] = files.progress_apply(
        lambda r: get_embedding_list(PropositionDataset(r['sentences_en']), tokenizer, model, device),
        axis=1
    )

    os.makedirs(save_path, exist_ok=True)
    np.save(
        os.path.join(save_path, "embeddings_sentences_en.npy"),
        files['embeddings_sentences_en'].to_numpy()
    )
    logging.info("Sentence embeddings saved successfully.")


def generate_paragraph_embeddings(files, save_path):
    logging.info("Generating paragraph embeddings...")
    tokenizer, model, device = init_model()
    tqdm.pandas(desc="Getting embeddings for paragraphs_formatted")

    files['embeddings_paragraphs_formatted'] = files.progress_apply(
        lambda r: get_embedding_list(PropositionDataset(r['paragraphs_formatted']), tokenizer, model, device),
        axis=1
    )

    os.makedirs(save_path, exist_ok=True)
    np.save(
        os.path.join(save_path, "embeddings_paragraphs_formatted.npy"),
        files['embeddings_paragraphs_formatted'].to_numpy()
    )
    logging.info("Paragraph embeddings saved successfully.")


def generate_proposition_embeddings(files, save_path):
    logging.info("Generating proposition embeddings...")
    tokenizer, model, device = init_model()
    tqdm.pandas(desc="Getting embeddings for propositions_en")

    files['embeddings_propositions_en'] = files.progress_apply(
        lambda r: get_embedding_list(PropositionDataset(r['propositions_en']), tokenizer, model, device),
        axis=1
    )

    os.makedirs(save_path, exist_ok=True)
    np.save(
        os.path.join(save_path, "embeddings_propositions_en.npy"),
        files['embeddings_propositions_en'].to_numpy()
    )
    logging.info("Proposition embeddings saved successfully.")


/home/user/Documents/ir/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Suppose 'files' is your DataFrame with columns: 
# ['sentences_en', 'paragraphs_formatted', 'propositions_en']

import pandas as pd
import os

# Suppose 'files' is your DataFrame
save_dir = "embeddings_output"

generate_proposition_embeddings(files, save_dir)

# Create a directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Save as pickle (preserves lists, dicts, etc.)
pickle_path = os.path.join(save_dir, "embeddings_output/embeddings_new_2_with_prop_afterprop.pkl")
files.to_pickle(pickle_path)

# To confirm save
print(f"DataFrame saved as pickle at: {pickle_path}")

Generating proposition embeddings...


2025-10-28 20:18:09.692931: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-28 20:18:10.038039: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-28 20:18:12.023834: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Getting embeddings for propositions_en: 100%|██████████| 9289/9289 [00:13<00:00, 692.00it/s

Proposition embeddings saved successfully.


OSError: Cannot save file into a non-existent directory: 'embeddings_output/embeddigs_output'

In [21]:

# Save as pickle (preserves lists, dicts, etc.)
pickle_path = os.path.join(save_dir, "embeddings_new_final_with_prop.pkl")
files.to_pickle(pickle_path)

# To confirm save
print(f"DataFrame saved as pickle at: {pickle_path}")

DataFrame saved as pickle at: embeddings_output/embeddings_new_final_with_prop.pkl


In [19]:
files

,filename,set,query,cases,text,paragraphs,paragraphs_formatted,suppressed_sections,sentences,sentences_en,...,sentences_en_set_list,paragraphs_formatted_en_set_list,judge,year,embeddings_paragraphs_formatted,embeddings_sentences_en,propositions,propositions_en,propositions_en_set_list,embeddings_propositions_en
0,079311,train,False,[],"Strayer, J.\n: These are two applications unde...","[[2] In April and May, 1993 the Department of ...","[In April and May, 1993 the Department of the ...",[],"[In April and May, 1993 the Department of the ...","[In April and May, 1993 the Department of the ...",...,"[{state, individu, translat, certain, depart, ...","[{indic, individu, depart, disclos, ernest, ac...",None,1994,"[[-0.20004287, 0.10326447, -0.025002336, 0.010...","[[-0.06782752, 0.026452757, -0.044306185, -0.2...",[],[],[],[]
1,015880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S...",[[1] : This is an application for a judicial r...,[: This is an application for a judicial revie...,[],[: This is an application for a judicial revie...,[: This is an application for a judicial revie...,...,"[{visa, perman, review, date, refus, resid, oc...","[{visa, perman, fact, review, date, refus, res...",None,2008,"[[0.49238092, 0.21510422, -0.42108464, 0.05929...","[[0.47733235, 0.2424333, -0.4347535, 0.0768362...",[],[],[],[]
2,042979,train,False,[],"Federal Court\nAngus Grant, for the applicant;...",[[2] The guiding principle in respect of new e...,[The guiding principle in respect of new evide...,[],[The guiding principle in respect of new evide...,[The guiding principle in respect of new evide...,...,"[{judith, state, risk, evid, assess, preremov,...","[{judith, state, risk, evid, assess, preremov,...",None,2009,"[[-0.11704859, 0.21959981, -0.21703193, 0.0462...","[[-0.109607965, 0.21841578, -0.19464128, -0.00...",[],[],[],[]
3,079847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...,[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[: This is an application for judicial review ...,[: This is an application for judicial review ...,...,"[{rsc, c, immigr, base, armenia, decis, backgr...","[{arriv, war, attack, fled, rsc, c, school, im...",None,2002,"[[0.01899644, 0.25205392, -0.3343874, 0.293352...","[[0.23801307, 0.06658855, -0.508649, 0.1040462...",[],[],[],[]
4,076089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU...",[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[: This is an application for judicial review ...,[: This is an application for judicial review ...,...,"[{subsect, render, c, immigr, sc, decis, alexa...","[{subsect, render, c, immigr, sc, decis, alexa...",None,2010,"[[0.13882422, 0.0042256033, -0.35483626, -0.03...","[[0.13882422, 0.0042256033, -0.35483626, -0.03...",[],[],[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9284,087439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ...","[[1] O'Reilly, J. : The Minister of National H...","[O'Reilly, J. : The Minister of National Healt...",[[6] A person who wishes to have a drug patent...,"[O'Reilly, J. : The Minister of National Healt...","[O'Reilly, J. : The Minister of National Healt...",...,"[{complianc, biovail, protect, notic, j, medic...","[{j, accord, minist, decis, regul, complianc, ...",O'Reilly,2005,"[[0.034605406, 0.31155354, -0.18228303, -0.428...","[[-0.1733626, 0.27614993, -0.08298564, -0.2654...",[A method of combining the active ingredients ...,[A method of combining the active ingredients ...,"[{substanc, creat, ingredi, case, activ, simil...","[[0.19184867, -0.14349103, 0.037140183, -0.261..."
9285,056791,test,False,[],"[1]\n: When this matter first came before me, ...","[[1] : When this matter first came before me, ...","[: When this matter first came before me, I he...",[],"[: When this matter first came before me, I he...","[: When this matter first ca

In [ ]:


import pandas as pd
import os

# Suppose 'files' is your DataFrame
save_dir = "embeddings_output"

generate_paragraph_embeddings(files, save_dir)

# Create a directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Save as pickle (preserves lists, dicts, etc.)
pickle_path = os.path.join(save_dir, "embeddings_new_2_with_prop_afterproppara.pkl")
files.to_pickle(pickle_path)

# To confirm save
print(f"DataFrame saved as pickle at: {pickle_path}")


Generating paragraph embeddings...


Getting embeddings for paragraphs_formatted:  50%|████▉     | 4615/9289 [7:07:35<11:53:56,  9.16s/it]

In [ ]:


import pandas as pd
import os

# Suppose 'files' is your DataFrame
save_dir = "embeddings_output"

generate_sentence_embeddings(files, save_dir)

# Create a directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Save as pickle (preserves lists, dicts, etc.)
pickle_path = os.path.join(save_dir, "embeddings_new_2_with_prop_afterpropparasent.pkl")
files.to_pickle(pickle_path)

# To confirm save
print(f"DataFrame saved as pickle at: {pickle_path}")


Generating sentence embeddings...


Getting embeddings for sentences_en: 100%|██████████| 9289/9289 [4:29:06<00:00,  1.74s/it]   


Sentence embeddings saved successfully.
DataFrame saved as pickle at: embeddings_output/embeddings_new_2_without_prop_afterpropparasent.pkl


In [ ]:
files.to_csv('final/embeddings_new_2_all_final_withprop.csv', index=False)
files

,filename,set,query,cases,text,paragraphs,paragraphs_formatted,suppressed_sections,propositions,propositions_en,...,sentences_en_string,sentences_en_set,sentences_en_set_list,paragraphs_formatted_en_set_list,propositions_en_set_list,judge,year,embeddings_propositions_en,embeddings_paragraphs_formatted,embeddings_sentences_en
0,79311,train,False,[],"Strayer, J.\n: These are two applications unde...","[[2] In April and May, 1993 the Department of ...","[In April and May, 1993 the Department of the ...",[],[],[],...,april depart secretari state receiv request in...,"{tender, distribut, industriel, offic, depart,...","[{certain, receiv, pertain, servic, request, i...","[{copi, basi, applic, ident, essenti, identifi...",[],None,1994,[],"[[-0.20004287, 0.10326447, -0.025002336, 0.010...","[[-0.06782752, 0.026452757, -0.044306185, -0.2..."
1,15880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S...",[[1] : This is an application for a judicial r...,[: This is an application for a judicial revie...,[],[],[],...,applic judici review decis visa offic date oct...,"{applic, notwithstand, skill, reli, point, all...","[{refus, applic, perman, offic, octob, review,...","[{refus, applic, perman, offic, octob, review,...",[],None,2008,[],"[[0.49238092, 0.21510422, -0.42108464, 0.05929...","[[0.47733235, 0.2424333, -0.4347535, 0.0768362..."
2,42979,train,False,[],"Federal Court\nAngus Grant, for the applicant;...",[[2] The guiding principle in respect of new e...,[The guiding principle in respect of new evide...,[],[],[],...,guid principl respect new evid submit preremov...,"{newli, lack, lesbian, offic, left, russel, ce...","[{submit, preremov, respect, justic, clearli, ...","[{submit, preremov, respect, justic, clearli, ...",[],None,2009,[],"[[-0.11704859, 0.21959981, -0.21703193, 0.0462...","[[-0.109607965, 0.21841578, -0.19464128, -0.00..."
3,79847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...,[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[],...,applic judici review decis convent refuge dete...,"{applic, juli, reli, relianc, area, declar, ga...","[{applic, juli, immigr, refuge, declar, gaspar...","[{continu, applic, juli, immigr, canada, refug...",[],None,2002,[],"[[0.01899644, 0.25205392, -0.3343874, 0.293352...","[[0.23801307, 0.06658855, -0.508649, 0.1040462..."
4,76089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU...",[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[],[],...,applic judici review pursuant subsect immigr r...,"{cruel, applic, legal, born, contrari, compari...","[{applic, immigr, subsect, refuge, sc, april, ...","[{applic, immigr, subsect, refuge, sc, april, ...",[],None,2010,[],"[[0.13882422, 0.0042256033, -0.35483626, -0.03...","[[0.13882422, 0.0042256033, -0.35483626, -0.03..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9284,87439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ...","[[1] O'Reilly, J. : The Minister of National H...","[O'Reilly, J. : The Minister of National Healt...",[[6] A person who wishes to have a drug patent...,[A person who wishes to have a drug patent pro...,[A person who wishes to have a drug patent pro...,...,oreilli j minist nation health welfar deni bio...,"{mainli, area, begin, plu, paramount, certifi,...","[{welfar, special, minist, j, afford, request,...","[{afford, medicin, biovail, inelig, minist, re...","[{come, reason, affect, deliveri, medicin, are...",O'Reilly,2005,"[[0.2033237, 0.07713017, -0.0013799854, -0.332...","[[0.034605406, 0.31155354, -0.18228303, -0.428...","[[-0.1733626, 0.27614993, -0.08298564, -0.2654..."
9285,56791,test,False,[],"[1]\n: When this matter first came before me, ...","[[1] : When this matter first came before me, ...","[: When this matter first came before me, I he...",[],[],[],...,matter came held sell deal certain line tobler...,"{profitda

In [2]:
import pickle


In [3]:
filename = 'embeddings_output/embeddings_new_2_with_prop_afterpropparasent.pkl'
files=None
with open(filename, 'rb') as f:
    files = pickle.load(f)
files 

,filename,set,query,cases,text,paragraphs,paragraphs_formatted,suppressed_sections,sentences,sentences_en,...,sentences_en_set_list,paragraphs_formatted_en_set_list,judge,year,embeddings_paragraphs_formatted,embeddings_sentences_en,propositions,propositions_en,propositions_en_set_list,embeddings_propositions_en
0,079311,train,False,[],"Strayer, J.\n: These are two applications unde...","[[2] In April and May, 1993 the Department of ...","[In April and May, 1993 the Department of the ...",[],"[In April and May, 1993 the Department of the ...","[In April and May, 1993 the Department of the ...",...,"[{state, individu, translat, certain, depart, ...","[{indic, individu, depart, disclos, ernest, ac...",None,1994,"[[-0.20004287, 0.10326447, -0.025002336, 0.010...","[[-0.06782752, 0.026452757, -0.044306185, -0.2...",[],[],[],[]
1,015880,train,False,[],"Federal Court\nFebruary 22, 2008. <FRAGMENT_S...",[[1] : This is an application for a judicial r...,[: This is an application for a judicial revie...,[],[: This is an application for a judicial revie...,[: This is an application for a judicial revie...,...,"[{visa, perman, review, date, refus, resid, oc...","[{visa, perman, fact, review, date, refus, res...",None,2008,"[[0.49238092, 0.21510422, -0.42108464, 0.05929...","[[0.47733235, 0.2424333, -0.4347535, 0.0768362...",[],[],[],[]
2,042979,train,False,[],"Federal Court\nAngus Grant, for the applicant;...",[[2] The guiding principle in respect of new e...,[The guiding principle in respect of new evide...,[],[The guiding principle in respect of new evide...,[The guiding principle in respect of new evide...,...,"[{judith, state, risk, evid, assess, preremov,...","[{judith, state, risk, evid, assess, preremov,...",None,2009,"[[-0.11704859, 0.21959981, -0.21703193, 0.0462...","[[-0.109607965, 0.21841578, -0.19464128, -0.00...",[],[],[],[]
3,079847,train,False,[],MLB unedited judgment <FRAGMENT_SUPPRESSED>\n...,[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[: This is an application for judicial review ...,[: This is an application for judicial review ...,...,"[{rsc, c, immigr, base, armenia, decis, backgr...","[{arriv, war, attack, fled, rsc, c, school, im...",None,2002,"[[0.01899644, 0.25205392, -0.3343874, 0.293352...","[[0.23801307, 0.06658855, -0.508649, 0.1040462...",[],[],[],[]
4,076089,train,False,[],"Federal Court\nJanuary 18, 2010. <FRAGMENT_SU...",[[1] : This is an application for judicial rev...,[: This is an application for judicial review ...,[],[: This is an application for judicial review ...,[: This is an application for judicial review ...,...,"[{subsect, render, c, immigr, sc, decis, alexa...","[{subsect, render, c, immigr, sc, decis, alexa...",None,2010,"[[0.13882422, 0.0042256033, -0.35483626, -0.03...","[[0.13882422, 0.0042256033, -0.35483626, -0.03...",[],[],[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9284,087439,test,True,[],"[1]\nO'Reilly, J.\n: The Minister of National ...","[[1] O'Reilly, J. : The Minister of National H...","[O'Reilly, J. : The Minister of National Healt...",[[6] A person who wishes to have a drug patent...,"[O'Reilly, J. : The Minister of National Healt...","[O'Reilly, J. : The Minister of National Healt...",...,"[{complianc, biovail, protect, notic, j, medic...","[{j, accord, minist, decis, regul, complianc, ...",O'Reilly,2005,"[[0.034605406, 0.31155354, -0.18228303, -0.428...","[[-0.1733626, 0.27614993, -0.08298564, -0.2654...",[A method of combining the active ingredients ...,[A method of combining the active ingredients ...,"[{substanc, creat, ingredi, case, activ, simil...","[[0.15676539, 0.07378287, -0.026583735, -0.063..."
9285,056791,test,False,[],"[1]\n: When this matter first came before me, ...","[[1] : When this matter first came before me, ...","[: When this matter first came before me, I he...",[],"[: When this matter first came before me, I he...","[: When this matter first ca

In [22]:
files['filename'] = files['filename'].astype(str).str.zfill(6)


In [5]:
filename = 'pickle/features_para_100_unique.pkl'
pairs=None
with open(filename, 'rb') as f:
    pairs = pickle.load(f)
pairs

,query,target,tuple,set,match,prop_max_cos_sim_sents,prop_max_cos_sim_paras,prop_max_jaccard_sents,prop_max_jaccard_paras,prop_max_overlap_sents,...,bin_0.00_0.10,bin_0.10_0.20,bin_0.20_0.30,bin_0.30_0.40,bin_0.40_0.50,bin_0.50_0.60,bin_0.60_0.70,bin_0.70_0.80,bin_0.80_0.90,bin_0.90_1.00
0,000002,051020,"(000002, 051020)",train,0,0.905936,0.931257,0.277778,0.375000,0.363636,...,0.0,0.0,0.0,0.0,0.000020,0.015124,0.336715,0.574397,0.073029,0.000714
1,000002,033359,"(000002, 033359)",train,0,0.915935,0.895876,0.279070,0.444444,0.325581,...,0.0,0.0,0.0,0.0,0.000000,0.007193,0.287434,0.582061,0.118908,0.004404
2,000002,078174,"(000002, 078174)",train,0,0.926757,0.934739,0.272727,0.233333,0.272727,...,0.0,0.0,0.0,0.0,0.000000,0.008940,0.270717,0.607283,0.109312,0.003748
3,000002,080156,"(000002, 080156)",train,0,0.889404,0.887285,0.233333,0.156863,0.318182,...,0.0,0.0,0.0,0.0,0.000225,0.020431,0.353390,0.551414,0.073706,0.000834
4,000002,006393,"(000002, 006393)",train,0,0.916897,0.873525,0.279070,0.432836,0.279070,...,0.0,0.0,0.0,0.0,0.000073,0.008870,0.272919,0.614031,0.101563,0.002545
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35079,099982,081592,"(099982, 081592)",test,None,0.919518,0.915024,0.142857,0.138614,0.210526,...,0.0,0.0,0.0,0.0,0.000362,0.048952,0.355549,0.492648,0.102088,0.000401
35080,099982,068882,"(099982, 068882)",test,None,0.894070,0.850566,0.097222,0.101266,0.210526,...,0.0,0.0,0.0,0.0,0.000193,0.034237,0.369292,0.525104,0.070767,0.000407
35081,099982,055811,"(099982, 055811)",test,None,0.907441,0.901313,0.133333,0.142857,0.240000,...,0.0,0.0,0.0,0.0,0.000160,0.022253,0.288416,0.541071,0.147345,0.000755
35082,099982,026079,"(099982, 026079)",test,None,0.900095,0.888118,0.156250,0.173333,0.263158,...,0.0,0.0,0.0,0.0,0.000000,0.001261,0.089824,0.625411,0.282648,0.000856


In [6]:
# Make a copy of the slice to avoid SettingWithCopyWarning
pairs = pairs[['query', 'target', 'tuple', 'set', 'match']].copy()

# Now modify safely
pairs['query'] = pairs['query'].astype(str).str.strip().str.zfill(6)
pairs['target'] = pairs['target'].astype(str).str.strip().str.zfill(6)


In [7]:
pairs

,query,target,tuple,set,match
0,000002,051020,"(000002, 051020)",train,0
1,000002,033359,"(000002, 033359)",train,0
2,000002,078174,"(000002, 078174)",train,0
3,000002,080156,"(000002, 080156)",train,0
4,000002,006393,"(000002, 006393)",train,0
...,...,...,...,...,...
35079,099982,081592,"(099982, 081592)",test,None
35080,099982,068882,"(099982, 068882)",test,None
35081,099982,055811,"(099982, 055811)",test,None
35082,099982,026079,"(099982, 026079)",test,None


In [13]:
files['filename'] = files['filename'].astype(str).str.zfill(6)


In [ ]:
import pandas as pd
import ast

def revert_dataframe_columns(df):
    for col in df.columns:
        def try_eval(x):
            if isinstance(x, str):
                try:
                    return ast.literal_eval(x)
                except (ValueError, SyntaxError):
                    return x
            return x
        df[col] = df[col].apply(try_eval)
    return df

files = pd.read_csv('final/embeddings_new_2_all_final.csv')
files = revert_dataframe_columns(files)


In [ ]:
files

In [23]:
import importlib, pairs_code

In [24]:
importlib.reload(pairs_code)
from pairs_code import (get_pairs, add_bins, get_prop_max_cos_sim_sents, get_prop_max_cos_sim_paras,
                        get_prop_max_jaccard_sents, get_prop_max_jaccard_paras, get_prop_max_overlap_sents, get_prop_max_overlap_paras, add_max_overall,
                        get_case_jaccard_sims, check_same_case, get_case_tfidf_scores, get_num_quotes, binarize_quotes, check_years, add_judge_checks)

In [25]:

# pairs = get_pairs(files)

In [26]:
pairs

,query,target,tuple,set,match
0,000002,051020,"(000002, 051020)",train,0
1,000002,033359,"(000002, 033359)",train,0
2,000002,078174,"(000002, 078174)",train,0
3,000002,080156,"(000002, 080156)",train,0
4,000002,006393,"(000002, 006393)",train,0
...,...,...,...,...,...
35079,099982,081592,"(099982, 081592)",test,None
35080,099982,068882,"(099982, 068882)",test,None
35081,099982,055811,"(099982, 055811)",test,None
35082,099982,026079,"(099982, 026079)",test,None


In [27]:
# Generate pairs dataframe. One query-candidate case pair per row. Compare file features from files df to generate pair features:

get_prop_max_cos_sim_sents(files, pairs)


Getting max cos similarity between query propositions and target sentences_en.


Getting max cosine similarity: 100%|██████████| 186158/186158 [03:45<00:00, 826.95it/s] 

Maximum proposition cosine similarity (with sentences_en) added to pairs.


In [28]:
get_prop_max_cos_sim_paras(files, pairs)


Getting max cos similarity between query propositions and target paragraphs_formatted.


Getting max cosine similarity: 100%|██████████| 186158/186158 [01:47<00:00, 1724.57it/s]

Maximum proposition cosine similarity (with paragraphs_formatted) added to pairs.


In [29]:
get_prop_max_jaccard_sents(files,pairs)


Getting max jaccard similarity between sentences_en_sets and propositions_en_sets.


Getting max jaccard proposition-sentences: 100%|██████████| 186158/186158 [10:06<00:00, 306.80it/s]

Added prop_max_jaccard_sents to pairs.


In [30]:
get_prop_max_jaccard_paras(files,pairs)


Getting max jaccard similarity between sentences_en_sets and paragraphs_formatted_set.


Getting max jaccard proposition-paragraphs: 100%|██████████| 186158/186158 [03:12<00:00, 966.29it/s] 

Added prop_max_jaccard_paras to pairs.


In [31]:
get_prop_max_overlap_sents(files,pairs)


Getting max overlap ratio between sentences_en_sets and propositions_en_sets.


Getting max overlap proposition-sentences: 100%|██████████| 186158/186158 [03:01<00:00, 1023.81it/s]

Added prop_max_overlap_sents to pairs.


In [32]:
get_prop_max_overlap_paras(files,pairs)


Getting max overlap ratio between paragraphs_formatted_set and propositions_en_sets.


Getting max overlap proposition-paragraphs: 100%|██████████| 186158/186158 [00:48<00:00, 3807.60it/s]

Added prop_max_overlap_paras to pairs.


In [33]:
add_max_overall(pairs,files)


Adding maximum overall proposition matching scores.


Adding max prop-sent cossim overall:   0%|          | 0/2057 [00:00<?, ?it/s]/home/user/Documents/ir/COLIEE_2024_Task1/pairs_code.py:570: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.9826549887657166' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pairs.at[target_indices[max_index], 'max_overall'] = max_score
Adding max prop-sent cossim overall: 100%|██████████| 2057/2057 [14:47<00:00,  2.32it/s]

Added max_overall to pairs.


In [34]:
get_case_jaccard_sims(files,pairs)


Getting jaccard similarity score between cases for both entity and word set.


Calculating sentences en set jaccard similarity: 100%|██████████| 186158/186158 [00:09<00:00, 19671.91it/s]

Added jaccard similarity scores in entity_set_jaccard and sentences_en_set_jaccard.


In [35]:
check_same_case(pairs)


Calculating sentences en set jaccard similarity: 100%|██████████| 186158/186158 [00:00<00:00, 272659.20it/s]

Add same_case check to pairs.


In [36]:
get_case_tfidf_scores(files,pairs)


Getting tfidf for entity_string


Calculating tfidf for entity_string: 100%|██████████| 186158/186158 [00:49<00:00, 3775.45it/s]

Getting tfidf for sentences_en_string



Calculating tfidf for sentences_en_string: 100%|██████████| 186158/186158 [00:53<00:00, 3477.09it/s]


Added tfidf scores to pairs.


In [37]:
get_num_quotes(files,pairs)


Get number of query quotes in target text.


Getting number of query quotes: 100%|██████████| 186158/186158 [06:21<00:00, 487.36it/s]


In [38]:
binarize_quotes(pairs)


Getting number of query quotes: 100%|██████████| 186158/186158 [00:00<00:00, 2593875.60it/s]

Added quotes to pairs.


In [39]:
check_years(files,pairs)


Checking if target case pre-dates query case year.


Checking years: 100%|██████████| 186158/186158 [00:00<00:00, 262802.12it/s]

Added check_year to pairs.


In [40]:
add_judge_checks(files,pairs)
add_bins(files, pairs)

Add judge pairs: 100%|██████████| 186158/186158 [00:00<00:00, 268247.02it/s]


Added judge_pair_ratio to pairs.
Added judge_match to pairs.
Generating histogram bins using cosine similarity from sentence_en embeddings.


Getting histogram bins: 100%|██████████| 186158/186158 [18:25<00:00, 168.45it/s]

Histogram bins added to "pairs" df.


In [43]:
pairs

,query,target,tuple,set,match,prop_max_cos_sim_sents,prop_max_cos_sim_paras,prop_max_jaccard_sents,prop_max_jaccard_paras,prop_max_overlap_sents,...,bin_0.00_0.10,bin_0.10_0.20,bin_0.20_0.30,bin_0.30_0.40,bin_0.40_0.50,bin_0.50_0.60,bin_0.60_0.70,bin_0.70_0.80,bin_0.80_0.90,bin_0.90_1.00
0,000002,051020,"(000002, 051020)",train,0,0.930569,0.936472,0.407407,0.392157,0.666667,...,0.0,0.0,0.0,0.0,0.000020,0.015124,0.336715,0.574397,0.073029,0.000714
1,000002,033359,"(000002, 033359)",train,0,0.982655,0.926958,0.461538,0.541667,0.500000,...,0.0,0.0,0.0,0.0,0.000000,0.007193,0.287434,0.582061,0.118908,0.004404
2,000002,078174,"(000002, 078174)",train,0,0.952116,0.953723,0.363636,0.263158,0.444444,...,0.0,0.0,0.0,0.0,0.000000,0.008940,0.270717,0.607283,0.109312,0.003748
3,000002,080156,"(000002, 080156)",train,0,0.913907,0.902539,0.263158,0.146341,0.555556,...,0.0,0.0,0.0,0.0,0.000225,0.020431,0.353390,0.551414,0.073706,0.000834
4,000002,006393,"(000002, 006393)",train,0,0.982655,0.908773,0.461538,0.490566,0.461538,...,0.0,0.0,0.0,0.0,0.000073,0.008870,0.272919,0.614031,0.101563,0.002545
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35079,099982,081592,"(099982, 081592)",test,None,0.888025,0.893385,0.166667,0.128205,0.272727,...,0.0,0.0,0.0,0.0,0.000362,0.048952,0.355549,0.492648,0.102088,0.000401
35080,099982,068882,"(099982, 068882)",test,None,0.879995,0.844341,0.157895,0.090909,0.300000,...,0.0,0.0,0.0,0.0,0.000193,0.034237,0.369292,0.525104,0.070767,0.000407
35081,099982,055811,"(099982, 055811)",test,None,0.891096,0.881440,0.150000,0.111111,0.363636,...,0.0,0.0,0.0,0.0,0.000160,0.022253,0.288416,0.541071,0.147345,0.000755
35082,099982,026079,"(099982, 026079)",test,None,0.899131,0.895012,0.166667,0.125000,0.363636,...,0.0,0.0,0.0,0.0,0.000000,0.001261,0.089824,0.625411,0.282648,0.000856


In [41]:
import pickle 
filename = 'pickle/pairs_bhavya_with_prop.pkl'

with open(filename, 'wb') as f:
    pickle.dump(pairs, f)
print("Saved pickle: ", filename) 

Saved pickle:  pickle/pairs_bhavya_with_prop.pkl


In [42]:
import pickle 
filename = 'pickle/pairs_bhavya_with_prop_2.pkl'

with open(filename, 'wb') as f:
    pickle.dump(pairs, f)
print("Saved pickle: ", filename) 

Saved pickle:  pickle/pairs_bhavya_with_prop_2.pkl


# model training from here

In [4]:
import pickle 
filename = 'pickle/features_para_100_unique.pkl'

with open(filename, 'rb') as f:
    pairs = pickle.load(f)

pairs

,query,target,tuple,set,match,prop_max_cos_sim_sents,prop_max_cos_sim_paras,prop_max_jaccard_sents,prop_max_jaccard_paras,prop_max_overlap_sents,...,bin_0.00_0.10,bin_0.10_0.20,bin_0.20_0.30,bin_0.30_0.40,bin_0.40_0.50,bin_0.50_0.60,bin_0.60_0.70,bin_0.70_0.80,bin_0.80_0.90,bin_0.90_1.00
0,000002,051020,"(000002, 051020)",train,0,0.905936,0.931257,0.277778,0.375000,0.363636,...,0.0,0.0,0.0,0.0,0.000020,0.015124,0.336715,0.574397,0.073029,0.000714
1,000002,033359,"(000002, 033359)",train,0,0.915935,0.895876,0.279070,0.444444,0.325581,...,0.0,0.0,0.0,0.0,0.000000,0.007193,0.287434,0.582061,0.118908,0.004404
2,000002,078174,"(000002, 078174)",train,0,0.926757,0.934739,0.272727,0.233333,0.272727,...,0.0,0.0,0.0,0.0,0.000000,0.008940,0.270717,0.607283,0.109312,0.003748
3,000002,080156,"(000002, 080156)",train,0,0.889404,0.887285,0.233333,0.156863,0.318182,...,0.0,0.0,0.0,0.0,0.000225,0.020431,0.353390,0.551414,0.073706,0.000834
4,000002,006393,"(000002, 006393)",train,0,0.916897,0.873525,0.279070,0.432836,0.279070,...,0.0,0.0,0.0,0.0,0.000073,0.008870,0.272919,0.614031,0.101563,0.002545
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35079,099982,081592,"(099982, 081592)",test,None,0.919518,0.915024,0.142857,0.138614,0.210526,...,0.0,0.0,0.0,0.0,0.000362,0.048952,0.355549,0.492648,0.102088,0.000401
35080,099982,068882,"(099982, 068882)",test,None,0.894070,0.850566,0.097222,0.101266,0.210526,...,0.0,0.0,0.0,0.0,0.000193,0.034237,0.369292,0.525104,0.070767,0.000407
35081,099982,055811,"(099982, 055811)",test,None,0.907441,0.901313,0.133333,0.142857,0.240000,...,0.0,0.0,0.0,0.0,0.000160,0.022253,0.288416,0.541071,0.147345,0.000755
35082,099982,026079,"(099982, 026079)",test,None,0.900095,0.888118,0.156250,0.173333,0.263158,...,0.0,0.0,0.0,0.0,0.000000,0.001261,0.089824,0.625411,0.282648,0.000856


In [5]:
import pandas as pd
pd.set_option('display.max_columns', None)  # show all columns
pd.set_option('display.width', None)        # don't wrap lines
pairs.head()  # or pairs

,query,target,tuple,set,match,prop_max_cos_sim_sents,prop_max_cos_sim_paras,prop_max_jaccard_sents,prop_max_jaccard_paras,prop_max_overlap_sents,prop_max_overlap_paras,max_overall,entity_set_jaccard,sentences_en_set_jaccard,same_case,tfidf_entity_string,tfidf_sentences_en_string,quotes_num,quotes_any,check_year,judge_pair,judge_pair_ratio,judge_match,bin_0.00_0.10,bin_0.10_0.20,bin_0.20_0.30,bin_0.30_0.40,bin_0.40_0.50,bin_0.50_0.60,bin_0.60_0.70,bin_0.70_0.80,bin_0.80_0.90,bin_0.90_1.00
0,000002,051020,"(000002, 051020)",train,0,0.905936,0.931257,0.277778,0.375000,0.363636,0.558140,0.942210,0.067797,0.252648,0,0.010430,0.085259,0,0,0,"(None, None)",57.965616,1,0.0,0.0,0.0,0.0,0.000020,0.015124,0.336715,0.574397,0.073029,0.000714
1,000002,033359,"(000002, 033359)",train,0,0.915935,0.895876,0.279070,0.444444,0.325581,0.651163,0.915762,0.136364,0.244807,0,0.225262,0.152766,1,1,0,"(None, Gleason)",1.002865,0,0.0,0.0,0.0,0.0,0.000000,0.007193,0.287434,0.582061,0.118908,0.004404
2,000002,078174,"(000002, 078174)",train,0,0.926757,0.934739,0.272727,0.233333,0.272727,0.476190,0.000000,0.183673,0.258873,0,0.369114,0.234738,0,0,1,"(None, None)",57.965616,1,0.0,0.0,0.0,0.0,0.000000,0.008940,0.270717,0.607283,0.109312,0.003748
3,000002,080156,"(000002, 080156)",train,0,0.889404,0.887285,0.233333,0.156863,0.318182,0.363636,0.914507,0.033708,0.261569,0,0.000681,0.078451,0,0,0,"(None, Roy)",0.458453,0,0.0,0.0,0.0,0.0,0.000225,0.020431,0.353390,0.551414,0.073706,0.000834
4,000002,006393,"(000002, 006393)",train,0,0.916897,0.873525,0.279070,0.432836,0.279070,0.674419,0.877102,0.056604,0.268229,0,0.006764,0.089809,1,1,0,"(None, None)",57.965616,1,0.0,0.0,0.0,0.0,0.000073,0.008870,0.272919,0.614031,0.101563,0.002545


In [6]:
# pairs = pairs.drop('bm25_score', axis=1)
pairs[pairs['query']=='062547']
# pairs[pairs['set']=='test']

,query,target,tuple,set,match,prop_max_cos_sim_sents,prop_max_cos_sim_paras,prop_max_jaccard_sents,prop_max_jaccard_paras,prop_max_overlap_sents,prop_max_overlap_paras,max_overall,entity_set_jaccard,sentences_en_set_jaccard,same_case,tfidf_entity_string,tfidf_sentences_en_string,quotes_num,quotes_any,check_year,judge_pair,judge_pair_ratio,judge_match,bin_0.00_0.10,bin_0.10_0.20,bin_0.20_0.30,bin_0.30_0.40,bin_0.40_0.50,bin_0.50_0.60,bin_0.60_0.70,bin_0.70_0.80,bin_0.80_0.90,bin_0.90_1.00
21197,062547,035691,"(062547, 035691)",test,None,0.965193,0.998340,0.739130,1.000000,0.739130,1.000000,0.964544,1.000000,1.000000,1,1.000000,1.000000,2,1,1,"(Tremblay-Lamer, Tremblay-Lamer)",0.114613,1,0.0,0.0,0.0,0.0,0.000000,0.001345,0.100448,0.585202,0.290135,0.022870
21198,062547,043567,"(062547, 043567)",test,None,0.935536,0.903004,0.210526,0.229885,0.347826,0.384615,0.000000,0.120000,0.239362,0,0.061881,0.281579,0,0,0,"(Tremblay-Lamer, Heald)",0.028653,0,0.0,0.0,0.0,0.0,0.000000,0.006836,0.202005,0.626752,0.160761,0.003646
21199,062547,091054,"(062547, 091054)",test,None,0.909170,0.904828,0.187500,0.161616,0.266667,0.333333,0.932454,0.082192,0.195219,0,0.026052,0.243845,0,0,0,"(Tremblay-Lamer, None)",1.604585,0,0.0,0.0,0.0,0.0,0.000196,0.009754,0.230558,0.554595,0.202540,0.002357
21200,062547,042386,"(062547, 042386)",test,None,0.938616,0.922668,0.404762,0.162791,0.447368,0.289474,0.000000,0.055046,0.148950,0,0.180631,0.122934,1,1,1,"(Tremblay-Lamer, None)",1.604585,0,0.0,0.0,0.0,0.0,0.000000,0.003690,0.167961,0.672730,0.154844,0.000775
21201,062547,038094,"(062547, 038094)",test,None,0.938386,0.921399,0.250000,0.396226,0.326923,0.423077,0.000000,0.153846,0.254941,0,0.239009,0.341830,1,1,1,"(Tremblay-Lamer, None)",1.604585,0,0.0,0.0,0.0,0.0,0.000000,0.003625,0.208316,0.586354,0.197015,0.004691
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21292,062547,066878,"(062547, 066878)",test,None,0.904189,0.884897,0.243243,0.148515,0.346154,0.277778,0.000000,0.039216,0.196226,0,0.003691,0.294120,0,0,0,"(Tremblay-Lamer, Hargrave)",0.444444,0,0.0,0.0,0.0,0.0,0.000086,0.018271,0.201578,0.583977,0.195145,0.000944
21293,062547,022699,"(062547, 022699)",test,None,0.890578,0.860256,0.171429,0.166667,0.304348,0.347826,0.937009,0.125000,0.229803,0,0.018073,0.269231,0,0,0,"(Tremblay-Lamer, None)",1.604585,0,0.0,0.0,0.0,0.0,0.000000,0.006099,0.242818,0.601509,0.146365,0.003210
21294,062547,092333,"(062547, 092333)",test,None,0.894662,0.896463,0.157895,0.130435,0.217391,0.304348,0.948838,0.115942,0.176923,0,0.049231,0.272471,0,0,0,"(Tremblay-Lamer, None)",1.604585,0,0.0,0.0,0.0,0.0,0.000035,0.009142,0.216678,0.628909,0.144267,0.000970
21295,062547,093389,"(062547, 093389)",test,None,0.885277,0.873027,0.121951,0.133333,0.266667,0.240741,0.000000,0.126984,0.216518,0,0.194904,0.287603,0,0,1,"(Tremblay-Lamer, McGillis)",0.444444,0,0.0,0.0,0.0,0.0,0.000000,0.001764,0.167088,0.650899,0.179009,0.001240


In [7]:
importlib.reload(pairs_code)
from pairs_code import (get_pairs, add_bins, get_prop_max_cos_sim_sents, get_prop_max_cos_sim_paras,
                        get_prop_max_jaccard_sents, get_prop_max_jaccard_paras, get_prop_max_overlap_sents, get_prop_max_overlap_paras, add_max_overall,
                        get_case_jaccard_sims, check_same_case, get_case_tfidf_scores, get_num_quotes, binarize_quotes, check_years, add_judge_checks)

In [11]:
# Do k-fold validation on train set to identify best hyperparameters:

importlib.reload(model_code)
from model_code import get_k_fold_model_dev_pairs, save_model_df_pairs

model_df_pairs = get_k_fold_model_dev_pairs(pairs)
save_model_df_pairs(model_df_pairs)

  0%|          | 0/4 [00:00<?, ?it/s]

Using train_queries: len 1245.000000 starting ['008459', '035369', '034511', '091987']
Using dev_queries  : len 415.000000 starting ['005628' '062926' '086288' '010578']
Judge matching based on k-fold train set only


/home/user/Documents/ir/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Trained MLP model 1/3
Trained MLP model 2/3
Trained MLP model 3/3
Trained RandomForest model 1/3
Trained RandomForest model 2/3
Trained RandomForest model 3/3
Returned trained ensemble model.


 25%|██▌       | 1/4 [03:38<10:55, 218.47s/it]

...

 75%|███████▌  | 3/4 [11:04<03:42, 222.30s/it]

Using train_queries: len 1245.000000 starting ['005628', '062926', '086288', '010578']
Using dev_queries  : len 415.000000 starting ['097158' '005862' '089555' '087161']
Judge matching based on k-fold train set only


/home/user/Documents/ir/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Trained MLP model 1/3
Trained MLP model 2/3
Trained MLP model 3/3
Trained RandomForest model 1/3
Trained RandomForest model 2/3
Trained RandomForest model 3/3
Returned trained ensemble model.


100%|██████████| 4/4 [14:44<00:00, 221.22s/it]


In [12]:
importlib.reload(model_code)
from model_code import apply_models_to_dfs
apply_models_to_dfs(model_df_pairs, infer_type=1)

Using infer type: 1
df: 0
Applying model to get probabilities:


Checking all target files: 100%|██████████| 1372/1372 [00:01<00:00, 841.63it/s]

num_predicted_class_1 1725
unique_query_predicted_class_1 414
unique_query_no_class_1 0
unique_target_predicted_class_1 1372
unique_target_no_class_1 0
average_query_class_1 4.166666666666667
Model performance on dev: 
Precision: 0.345507
Recall   : 0.641550
F1       : 0.449133
-------------------------------------
df: 1
Applying model to get probabilities:



Checking all target files: 100%|██████████| 1296/1296 [00:01<00:00, 876.87it/s]

num_predicted_class_1 1647
unique_query_predicted_class_1 412
unique_query_no_class_1 0
unique_target_predicted_class_1 1296
unique_target_no_class_1 0
average_query_class_1 3.9975728155339807
Model performance on dev: 
Precision: 0.335155
Recall   : 0.648649
F1       : 0.441954
-------------------------------------
df: 2
Applying model to get probabilities:



Checking all target files: 100%|██████████| 1315/1315 [00:01<00:00, 884.82it/s]

num_predicted_class_1 1687
unique_query_predicted_class_1 411
unique_query_no_class_1 0
unique_target_predicted_class_1 1315
unique_target_no_class_1 0
average_query_class_1 4.104622871046229
Model performance on dev: 
Precision: 0.349140
Recall   : 0.676234
F1       : 0.460516
-------------------------------------
df: 3
Applying model to get probabilities:



Checking all target files: 100%|██████████| 1291/1291 [00:01<00:00, 882.88it/s]

num_predicted_class_1 1648
unique_query_predicted_class_1 413
unique_query_no_class_1 0
unique_target_predicted_class_1 1291
unique_target_no_class_1 0
average_query_class_1 3.990314769975787
Model performance on dev: 
Precision: 0.320388
Recall   : 0.629321
F1       : 0.424608
-------------------------------------

==RESULTS=============================
(0.34550724637681157, 0.6415500538213132, 0.44913338357196686)
(0.33515482695810567, 0.6486486486486487, 0.4419535628502802)
(0.34914048606994663, 0.6762342135476463, 0.46051602814698983)
(0.32038834951456313, 0.6293206197854588, 0.42460796139927626)
Means:
(0.33754772722985676, 0.6489383839507668, 0.4440527339921283)


In [13]:
importlib.reload(model_code)
from model_code import apply_models_to_dfs
apply_models_to_dfs(model_df_pairs, infer_type=2)

Using infer type: 2
df: 0
Applying model to get probabilities:


Assigning queries: 100%|██████████| 12680/12680 [00:00<00:00, 70021.95it/s]


num_predicted_class_1 1500
unique_query_predicted_class_1 414
unique_query_no_class_1 0
unique_target_predicted_class_1 1372
unique_target_no_class_1 0
average_query_class_1 3.6231884057971016
Model performance on dev: 
Precision: 0.370667
Recall   : 0.598493
F1       : 0.457802
-------------------------------------
df: 1
Applying model to get probabilities:


Assigning queries: 100%|██████████| 11936/11936 [00:00<00:00, 69951.95it/s]

num_predicted_class_1 1430
unique_query_predicted_class_1 412
unique_query_no_class_1 0
unique_target_predicted_class_1 1296
unique_target_no_class_1 0
average_query_class_1 3.470873786407767
Model performance on dev: 
Precision: 0.367832
Recall   : 0.618096
F1       : 0.461201
-------------------------------------
df: 2
Applying model to get probabilities:



Assigning queries: 100%|██████████| 12016/12016 [00:00<00:00, 70132.40it/s]

num_predicted_class_1 1461
unique_query_predicted_class_1 411
unique_query_no_class_1 0
unique_target_predicted_class_1 1315
unique_target_no_class_1 0
average_query_class_1 3.5547445255474455
Model performance on dev: 
Precision: 0.381246
Recall   : 0.639495
F1       : 0.477702
-------------------------------------
df: 3
Applying model to get probabilities:



Assigning queries: 100%|██████████| 12063/12063 [00:00<00:00, 68974.30it/s]

num_predicted_class_1 1433
unique_query_predicted_class_1 413
unique_query_no_class_1 0
unique_target_predicted_class_1 1291
unique_target_no_class_1 0
average_query_class_1 3.469733656174334
Model performance on dev: 
Precision: 0.348221
Recall   : 0.594756
F1       : 0.439261
-------------------------------------

==RESULTS=============================
(0.37066666666666664, 0.5984930032292788, 0.4578015644298065)
(0.3678321678321678, 0.618096357226792, 0.46120122753178433)
(0.3812457221081451, 0.6394948335246843, 0.4777015437392796)
(0.34822051639916257, 0.5947556615017878, 0.4392605633802817)
Means:
(0.3669912682515355, 0.6127099638706357, 0.45899122477028803)


In [15]:
# Train model

importlib.reload(model_code)
from model_code import build_train_ensemble

train_df = pairs[pairs['set']=='train']
model, train_df = build_train_ensemble(train_df)

/home/user/Documents/ir/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Trained MLP model 1/3
Trained MLP model 2/3
Trained MLP model 3/3
Trained RandomForest model 1/3
Trained RandomForest model 2/3
Trained RandomForest model 3/3
Returned trained ensemble model.


In [16]:
# Generate final results

importlib.reload(model_code)
from model_code import inference_on_test

test_df = pairs[pairs['set']=='test']

for infer_type in [1,2]:
    results_df = inference_on_test(model, test_df, infer_type)
    print()

Applying model to get probabilities:


Checking all target files: 100%|██████████| 1624/1624 [00:03<00:00, 485.35it/s]


{'num_predicted_class_1': 1944, 'unique_query_predicted_class_1': 398, 'unique_query_no_class_1': 0, 'unique_target_predicted_class_1': 1624, 'unique_target_no_class_1': 0, 'average_query_class_1': 4.884422110552764}
Written results to file ./output/results_1.txt
Total number of lines: 1944
Number of unique strings in the first position: 398
Number of unique strings in the second position: 1624
Average number of lines per unique first position: 4.884422110552764

Applying model to get probabilities:


Assigning queries: 100%|██████████| 35084/35084 [00:00<00:00, 72610.40it/s]


{'num_predicted_class_1': 1747, 'unique_query_predicted_class_1': 398, 'unique_query_no_class_1': 0, 'unique_target_predicted_class_1': 1624, 'unique_target_no_class_1': 0, 'average_query_class_1': 4.389447236180905}
Written results to file ./output/results_2.txt
Total number of lines: 1747
Number of unique strings in the first position: 398
Number of unique strings in the second position: 1624
Average number of lines per unique first position: 4.389447236180905



In [17]:
pairs[(pairs['set']=='train') & (pairs['match']==1)]

,query,target,tuple,set,match,prop_max_cos_sim_sents,prop_max_cos_sim_paras,prop_max_jaccard_sents,prop_max_jaccard_paras,prop_max_overlap_sents,prop_max_overlap_paras,max_overall,entity_set_jaccard,sentences_en_set_jaccard,same_case,tfidf_entity_string,tfidf_sentences_en_string,quotes_num,quotes_any,check_year,judge_pair,judge_pair_ratio,judge_match,bin_0.00_0.10,bin_0.10_0.20,bin_0.20_0.30,bin_0.30_0.40,bin_0.40_0.50,bin_0.50_0.60,bin_0.60_0.70,bin_0.70_0.80,bin_0.80_0.90,bin_0.90_1.00
102,000031,013059,"(000031, 013059)",train,1,0.932408,0.936893,0.476190,0.458333,0.526316,0.714286,0.915085,0.118644,0.296948,0,0.122611,0.395665,1,1,1,"(None, Lafrenière)",0.229226,0,0.0,0.0,0.0,0.0,0.000000,0.012737,0.186812,0.610919,0.184092,0.005440
103,000031,074681,"(000031, 074681)",train,1,0.959820,0.971206,0.416667,0.500000,0.578947,0.842105,0.946126,0.218750,0.325949,0,0.224135,0.438879,3,1,1,"(None, None)",57.965616,1,0.0,0.0,0.0,0.0,0.000116,0.011070,0.169527,0.584725,0.230010,0.004552
104,000031,070067,"(000031, 070067)",train,1,0.943362,0.979263,0.416667,0.674699,0.461538,0.717949,0.000000,0.157143,0.353659,0,0.054354,0.455889,1,1,1,"(None, None)",57.965616,1,0.0,0.0,0.0,0.0,0.000000,0.010306,0.164195,0.592177,0.225278,0.008044
106,000031,091216,"(000031, 091216)",train,1,0.951308,0.930311,0.571429,0.315789,0.666667,0.833333,0.888517,0.133333,0.270891,0,0.042420,0.221847,1,1,1,"(None, None)",57.965616,1,0.0,0.0,0.0,0.0,0.000037,0.019773,0.307661,0.593971,0.077347,0.001212
108,000031,011771,"(000031, 011771)",train,1,0.919859,0.906254,0.250000,0.196429,0.500000,0.500000,0.909160,0.136364,0.258106,0,0.070190,0.153514,0,0,1,"(None, None)",57.965616,1,0.0,0.0,0.0,0.0,0.000020,0.012357,0.271597,0.621488,0.093782,0.000755
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150779,099856,046441,"(099856, 046441)",train,1,0.906456,0.890744,0.238095,0.155556,0.304348,0.347826,0.000000,0.187500,0.270115,0,0.328234,0.405537,0,0,1,"(None, None)",57.965616,1,0.0,0.0,0.0,0.0,0.000000,0.007900,0.201552,0.523839,0.247623,0.019086
150794,099856,079743,"(099856, 079743)",train,1,0.903363,0.887519,0.208333,0.178571,0.304348,0.391304,0.000000,0.035714,0.236878,0,0.000000,0.374195,0,0,1,"(None, None)",57.965616,1,0.0,0.0,0.0,0.0,0.000000,0.006815,0.213113,0.471038,0.287248,0.021786
150877,099892,097422,"(099892, 097422)",train,1,0.917842,0.946067,0.462963,0.540984,0.520833,0.650000,0.914913,0.145455,0.271752,0,0.151719,0.330151,0,0,1,"(None, None)",57.965616,1,0.0,0.0,0.0,0.0,0.000000,0.001639,0.129268,0.627034,0.239764,0.002295
150952,099892,035289,"(099892, 035289)",train,1,0.925443,0.941214,0.214286,0.353659,0.238095,0.460317,0.925443,0.076923,0.255569,0,0.001115,0.187779,1,1,1,"(None, None)",57.965616,1,0.0,0.0,0.0,0.0,0.000037,0.016231,0.223022,0.552575,0.204403,0.003731


In [ ]:

test_df = pairs[pairs['set']=='test']

In [ ]:
test_df

,query,target,tuple,set,match
2570,000140,007603,"(000140, 007603)",test,NaN
2571,000140,061981,"(000140, 061981)",test,NaN
2572,000140,018972,"(000140, 018972)",test,NaN
2573,000140,088946,"(000140, 088946)",test,NaN
2574,000140,057846,"(000140, 057846)",test,NaN
...,...,...,...,...,...
624010,099982,032076,"(099982, 032076)",test,NaN
624011,099982,087440,"(099982, 087440)",test,NaN
624012,099982,075123,"(099982, 075123)",test,NaN
624013,099982,071484,"(099982, 071484)",test,NaN


In [11]:
import pandas as pd

# assuming 'pairs' is your dataframe
pairs_new = test_df[['query', 'target']].copy()

# add a new column with the constant value
pairs_new['label'] = 'UMNLP1'

# display the new dataframe
print(pairs_new.head())

# if you want to save it as a space-separated text file:
pairs_new.to_csv('experiments/test_prediction_bm25.txt', sep=' ', header=False, index=False)


       query  target   label
2570  000140  007603  UMNLP1
2571  000140  061981  UMNLP1
2572  000140  018972  UMNLP1
2573  000140  088946  UMNLP1
2574  000140  057846  UMNLP1


In [12]:
pairs_new


,query,target,label
2570,000140,007603,UMNLP1
2571,000140,061981,UMNLP1
2572,000140,018972,UMNLP1
2573,000140,088946,UMNLP1
2574,000140,057846,UMNLP1
...,...,...,...
624010,099982,032076,UMNLP1
624011,099982,087440,UMNLP1
624012,099982,075123,UMNLP1
624013,099982,071484,UMNLP1


In [13]:

train_df = pairs[pairs['set']=='train']

In [14]:
import pandas as pd

# assuming 'pairs' is your dataframe
pairs_new = train_df[['query', 'target']].copy()

# add a new column with the constant value
pairs_new['label'] = 'UMNLP1'

# display the new dataframe
print(pairs_new.head())

# if you want to save it as a space-separated text file:
pairs_new.to_csv('experiments/train_prediction_bm25.txt', sep=' ', header=False, index=False)


    query  target   label
0  000002  054392  UMNLP1
1  000002  054431  UMNLP1
2  000002  072955  UMNLP1
3  000002  093012  UMNLP1
4  000002  044060  UMNLP1


In [15]:
import json

# Load the JSON file
json_train_labels_path = './data/task1_train_labels_2025.json'

# Open and load the file
with open(json_train_labels_path, 'r') as file:
    labels = json.load(file)

# Check if the key exists in the dictionary
key = '062547.txt'
if key in labels:
    print(f"The key '{key}' exists in the labels dictionary.")
else:
    print(f"The key '{key}' is NOT found in the labels dictionary.")


The key '062547.txt' is NOT found in the labels dictionary.
